# Hypothesis Generation RAG – Demo

Этот ноутбук демонстрирует использование **HypothesisGenerator** для генерации гипотез по материалам и процессам на основе базы знаний.

## 1. Импорт и настройка

In [ ]:
import sys
sys.path.append('..')  # Добавляем корневую папку проекта

from src.pipeline import HypothesisGenerator
from src.schemas import QueryRequest
import json
import os

## 2. Загрузка индекса

Если индекс уже построен (например, через скрипт `build_index.py`), загружаем его.

Если индекса нет, можно построить его прямо здесь (раскомментируйте строку ниже) — но обычно это делается один раз отдельно.

In [ ]:
index_path = "../faiss_index"  # путь к сохранённому индексу

# Если индекс отсутствует, создаём его из данных (занимает время):
# generator = HypothesisGenerator(knowledge_root="../data", index_path=index_path)
# Но предполагаем, что индекс уже есть, поэтому просто загружаем:

generator = HypothesisGenerator(index_path=index_path)
print("✅ Индекс загружен.")

## 3. Создание запроса (пример)

Заполняем структуру `QueryRequest` – цель, KPI, ограничения.

In [ ]:
query = QueryRequest(
    target="Увеличить жаропрочность никелевого суперсплава при 1000°C на 20%",
    kpi_name="Скорость ползучести",
    kpi_baseline=1e-5,
    kpi_target=8e-6,
    constraints={
        "raw_materials": ["Ni", "Co", "Al", "Ti", "Ta"],
        "budget": "200k USD",
        "equipment": ["вакуумно-дуговой переплав", "горячее изостатическое прессование"],
        "regulatory": ["ASTM E139"]
    },
    top_k_hypotheses=3
)

print("✅ Запрос создан:")
print(f"Цель: {query.target}")
print(f"KPI: {query.kpi_name} (база: {query.kpi_baseline}, цель: {query.kpi_target})")

## 4. Генерация гипотез

Запускаем пайплайн: многопоточный поиск → ранжирование → генерация.

In [ ]:
print("⏳ Генерация гипотез...")
response = generator.generate(query)
print("✅ Готово.")

## 5. Вывод результатов

Каждая гипотеза содержит:
- `statement` – формулировка
- `justification` – обоснование из найденных документов
- `mechanism` – предложенный физико-химический механизм
- `novelty` – оценка новизны
- `risk` – потенциальные риски
- `expected_kpi_impact` – ожидаемое влияние на целевой показатель
- `experimental_roadmap` – (опционально) дорожная карта экспериментов

In [ ]:
for hyp in response.hypotheses:
    print(f"🔬 ГИПОТЕЗА {hyp.rank}: {hyp.statement}")
    print(f"   Обоснование: {hyp.justification}")
    print(f"   Механизм: {hyp.mechanism}")
    print(f"   Новизна: {hyp.novelty}")
    print(f"   Риски: {hyp.risk}")
    print(f"   Ожидаемое влияние на KPI: {hyp.expected_kpi_impact}")
    if hyp.experimental_roadmap:
        print(f"   Дорожная карта: {hyp.experimental_roadmap}")
    print("-" * 80)

## 6. Сохранение результата в JSON

Для дальнейшего анализа или интеграции можно сохранить ответ в структурированном виде.

In [ ]:
output_path = "../hypotheses_output.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(response.dict(), f, indent=2, ensure_ascii=False)
print(f"✅ Результаты сохранены в {output_path}")